# Cell 1 — Install dependencies

In [ ]:
!pip install unsloth "trl>=0.12.0" datasets transformers peft accelerate bitsandbytes

import unsloth
import trl
import datasets
import transformers
import peft
import accelerate

print(f"Unsloth version: {unsloth.__version__}")
print(f"TRL version: {trl.__version__}")
print(f"Datasets version: {datasets.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"PEFT version: {peft.__version__}")
print(f"Accelerate version: {accelerate.__version__}")

# Cell 2 — Imports and config

In [ ]:
import os
import sys
import json
from datasets import load_from_disk

ENV_URL = os.environ.get("ENV_URL", "http://localhost:7860")
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 1400
LORA_R = 16
OUTPUT_DIR = "../evaluation/checkpoints/"
FINAL_ADAPTER_DIR = "../evaluation/checkpoints/final_adapter/"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)

# Cell 3 — Load model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none"
)

model.print_trainable_parameters()

# Cell 4 — Load dataset

In [ ]:
dataset = load_from_disk("../evaluation/grpo_dataset/")
print(f"Dataset loaded with {len(dataset)} items.\n")
print("Sample row:")
print(dataset[0])

# Cell 5 — Define reward functions

In [ ]:
sys.path.insert(0, "../")
from training.reward_fn import compute_reward, compute_format_reward

# Create a precise lookup dictionary mapping the prompt's string representation back to the actual PR ID payload.
# TRL passes unmodified 'prompts' directly to the reward function from the HF dataset.
prompt_to_pr = {}
for item in dataset:
    prompt_str = str(item["prompt"])
    prompt_to_pr[prompt_str] = item["pr_id"]

def reward_fn(completions, prompts, **kwargs):
    pr_ids = [prompt_to_pr.get(str(p), "unknown") for p in prompts]
    return compute_reward(completions, pr_ids, **kwargs)

def format_reward_fn(completions, **kwargs):
    return compute_format_reward(completions, **kwargs)

# Test call on a dummy payload
dummy_completion = ["{\"review_decision\": \"approve\"}"]
print("Format logic test score:", format_reward_fn(dummy_completion))

# Cell 6 — Configure and run GRPOTrainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_generations=4,
    max_prompt_length=1024,
    max_completion_length=300,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    max_steps=400,
    learning_rate=5e-6,
    logging_steps=5,
    save_steps=50,
    warmup_steps=20,
    report_to="none"
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn, format_reward_fn],
    args=config,
    train_dataset=dataset
)

trainer.train()

# Cell 7 — Save results

In [ ]:
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

os.makedirs("../evaluation/plots/", exist_ok=True)
with open("../evaluation/plots/training_log.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)

log_history = trainer.state.log_history
rewards = [log.get("reward", 0) for log in log_history if "reward" in log.keys()]
last_10 = rewards[-10:] if len(rewards) >= 10 else rewards

if last_10:
    mean_val = sum(last_10) / len(last_10)
    print(f"Final mean reward (last {len(last_10)} entries): {mean_val:.4f}")
else:
    print("No valid rewards discovered in training history logs.")